# 03 - Load the SME defect playbook index

Build plane, playbook step of the `msd_training` job.

* Source of truth: `src/playbook/defect_playbook.yaml` (versioned in Git, reviewed by integrity
  engineers through pull requests).
* This notebook **validates** the YAML (one entry per NEU class, required fields present and
  non-empty, unique ids, valid `base_severity`) and **overwrites** the Delta table
  `defect_playbook`, which the app reads at inference time to ground the LLM.
* The run fails if validation fails, so an invalid playbook never reaches the app.

In [0]:
import os

dbutils.widgets.text("catalog", "poc_metal_surface_defect_diagnostics")
dbutils.widgets.text("schema", "metal_surface_defects")
# Empty -> resolve relative to this notebook (../playbook/defect_playbook.yaml)
dbutils.widgets.text("playbook_path", "")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
playbook_path = dbutils.widgets.get("playbook_path") or os.path.normpath(
    os.path.join(os.getcwd(), "..", "playbook", "defect_playbook.yaml")
)
playbook_table = f"{catalog}.{schema}.defect_playbook"

CLASSES = ["crazing", "inclusion", "patches", "pitted_surface", "rolled-in_scale", "scratches"]
SEVERITIES = {"Low", "Medium", "High"}
TEXT_FIELDS = [
    "playbook_id", "defect_class", "display_name", "category", "base_severity", "description",
    "risk_assessment", "progression_if_untreated", "monitoring",
]
LIST_FIELDS = [
    "visual_indicators", "typical_root_causes", "recommended_inspection",
    "recommended_repair", "escalation_criteria",
]

print(f"playbook: {playbook_path}\ntable:    {playbook_table}")

In [0]:
import yaml

with open(playbook_path) as f:
    doc = yaml.safe_load(f)

errors = []
for key in ("playbook_version", "status", "approved_by", "last_reviewed", "entries"):
    if not doc.get(key):
        errors.append(f"missing top-level field '{key}'")

entries = doc.get("entries") or []
for e in entries:
    eid = e.get("playbook_id", "<no id>")
    for k in TEXT_FIELDS:
        if not isinstance(e.get(k), str) or not e[k].strip():
            errors.append(f"{eid}: '{k}' must be a non-empty string")
    for k in LIST_FIELDS:
        v = e.get(k)
        if not isinstance(v, list) or not v or not all(isinstance(i, str) and i.strip() for i in v):
            errors.append(f"{eid}: '{k}' must be a non-empty list of strings")
    if e.get("base_severity") not in SEVERITIES:
        errors.append(f"{eid}: base_severity must be one of {sorted(SEVERITIES)}")

classes = [e.get("defect_class") for e in entries]
ids = [e.get("playbook_id") for e in entries]
if sorted(classes) != sorted(CLASSES):
    errors.append(f"classes {sorted(classes)} do not match the NEU classes {sorted(CLASSES)}")
if len(set(ids)) != len(ids):
    errors.append("duplicate playbook_id values")

if errors:
    raise ValueError("Playbook validation failed:\n- " + "\n- ".join(errors))
print(f"OK - playbook v{doc['playbook_version']} ({doc['status']}), {len(entries)} entries")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StringType, StructField, StructType

schema_def = StructType(
    [StructField(k, StringType(), False) for k in TEXT_FIELDS]
    + [StructField(k, ArrayType(StringType()), False) for k in LIST_FIELDS]
    + [StructField(k, StringType(), False) for k in
       ("playbook_version", "playbook_status", "approved_by", "last_reviewed", "disclaimer")]
)

rows = [
    tuple(e[k].strip() for k in TEXT_FIELDS)
    + tuple([i.strip() for i in e[k]] for k in LIST_FIELDS)
    + (str(doc["playbook_version"]), str(doc["status"]), str(doc["approved_by"]),
       str(doc["last_reviewed"]), str(doc.get("disclaimer", "")).strip())
    for e in entries
]

df = spark.createDataFrame(rows, schema_def).withColumn("loaded_at", F.current_timestamp())
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(playbook_table)
spark.sql(
    f"COMMENT ON TABLE {playbook_table} IS 'SME defect playbook index (one row per NEU class), "
    "loaded from src/playbook/defect_playbook.yaml. Grounding source for the diagnostic report.'"
)

display(spark.table(playbook_table).select("playbook_id", "defect_class", "category", "base_severity", "playbook_version"))

In [0]:
pb = spark.table(playbook_table)
assert pb.count() == len(CLASSES), f"Expected {len(CLASSES)} rows, got {pb.count()}"
null_checks = pb.select(
    *[F.sum(F.col(k).isNull().cast("int")).alias(k) for k in TEXT_FIELDS + LIST_FIELDS]
).first().asDict()
assert not any(null_checks.values()), f"Null key fields: {null_checks}"
print(f"OK - {playbook_table}: {pb.count()} rows, no null key fields")

In [0]:
# Feedback / expert-review store written by the app (one row per submission). Created if missing and
# never overwritten: rows are human labels for model monitoring and future retraining.
feedback_table = f"{catalog}.{schema}.diagnosis_feedback"
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {feedback_table} (
  feedback_id           STRING    NOT NULL COMMENT 'UUID of the submission',
  submitted_at          TIMESTAMP NOT NULL,
  submitted_by          STRING    COMMENT 'App user email (X-Forwarded-Email)',
  feedback_type         STRING    NOT NULL COMMENT 'rating | review_request | expert_review',
  rating                STRING    COMMENT 'up | down (rating only)',
  verdict               STRING    COMMENT 'confirmed | corrected | inconclusive (expert_review only)',
  corrected_class       STRING    COMMENT 'Defect class according to the reviewer',
  corrected_severity    STRING    COMMENT 'Severity according to the reviewer',
  notes                 STRING,
  image_sha256          STRING,
  image_file_name       STRING,
  image_path            STRING    COMMENT 'Copy of the image in the raw volume (reviews/...)',
  predicted_class       STRING,
  confidence            DOUBLE,
  probabilities         STRING    COMMENT 'JSON class -> probability',
  severity              STRING    COMMENT 'Severity shown by the app',
  human_review_required BOOLEAN,
  review_reasons        STRING,
  playbook_refs         STRING    COMMENT 'Source line of the report, e.g. PB-NEU-PIT v1.0.0',
  classifier_endpoint   STRING,
  report_text           STRING,
  case_id               STRING    COMMENT 'Analysis case: links a review_request to its expert_review'
)
COMMENT 'Human feedback and expert reviews submitted from the Metal Surface Defect Diagnostics app.'
""")
# Additive schema evolution for tables created before a column existed.
if "case_id" not in spark.table(feedback_table).columns:
    spark.sql(f"ALTER TABLE {feedback_table} ADD COLUMNS (case_id STRING COMMENT "
              "'Analysis case: links a review_request to its expert_review')")
print(f"OK - {feedback_table} ({spark.table(feedback_table).count()} rows)")